# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfurqan1/FlyRank-MachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Prepare the Week 4 analysis frame

The baseline makes its decision using March 2026 search-performance signals. April impressions are used only afterward to evaluate whether a content item declined by more than 15%. No April or label-derived value will be used in the baseline score.

In [12]:
import getpass
from pathlib import Path
import json

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

# Retrieve the private token from Colab Secrets.
# In Colab's key panel, create a secret named HF_TOKEN
# and enable notebook access.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

# Hidden fallback; the token will not appear in notebook output.
if not hf_token:
    hf_token = getpass.getpass(
        "Enter your Hugging Face READ token: "
    )

if not hf_token.startswith("hf_"):
    raise ValueError(
        "A valid Hugging Face READ token is required."
    )

con = duckdb.connect()

safe_token = hf_token.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET flyrank_hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

# Remove the token variables after registering the DuckDB secret.
del hf_token, safe_token

WAREHOUSE_ROOT = (
    "hf://datasets/FlyRank/internship-warehouse"
)

MARCH_FACT = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

APRIL_FACT = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    "month=2026-04/*.parquet"
)

print("Warehouse connection configured.")
print("Observation window: March 2026")
print("Outcome window: April 2026")
print("Sealed month: June 2026")

Warehouse connection configured.
Observation window: March 2026
Outcome window: April 2026
Sealed month: June 2026


In [13]:
analysis_df = con.execute(
    f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0) AS march_avg_position
        FROM read_parquet('{MARCH_FACT}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM read_parquet('{APRIL_FACT}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_impressions,
        m.march_clicks,
        m.march_avg_position,
        100.0 * m.march_clicks
            / NULLIF(m.march_impressions, 0) AS march_ctr,
        a.april_impressions,
        CASE
            WHEN a.april_impressions < 0.85 * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march AS m
    INNER JOIN april AS a
        USING (client_hash_id, content_hash_id)
    WHERE m.march_impressions > 0
    """
).df()

print(f"Eligible March–April content items: {len(analysis_df):,}")
print(f"Observed April decline base rate: {analysis_df['is_declining_label'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible March–April content items: 158,549
Observed April decline base rate: 51.1%


### Signal 1 — March impression volume

The assignment identifies search volume as a signal behind FlyRank's quick-win flag logic: recommendations with more impressions may represent greater potential impact. I bucket March impressions using fixed, readable thresholds and compare each bucket with the subsequent April decline rate.

This test asks whether March volume is associated with the future outcome. It does not assume that higher volume causes decline.

In [14]:
volume_labels = ["1-99", "100-499", "500-2,999", "3,000+"]

analysis_df["volume_bucket"] = pd.cut(
    analysis_df["march_impressions"],
    bins=[0, 100, 500, 3000, np.inf],
    labels=volume_labels,
    right=False,
)

volume_signal_table = (
    analysis_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_march_impressions=("march_impressions", "median"),
        april_decline_rate=("is_declining_label", "mean"),
    )
    .reset_index()
)

volume_signal_table["april_decline_rate"] = (
    volume_signal_table["april_decline_rate"] * 100
).round(1)

print("Signal 1 — March impression volume and subsequent decline")
display(volume_signal_table)

Signal 1 — March impression volume and subsequent decline


,volume_bucket,n,median_march_impressions,april_decline_rate
0,1-99,57656,20.0,43.6
1,100-499,39047,227.0,57.0
2,"500-2,999",39694,1139.0,56.2
3,"3,000+",22152,5919.0,50.8


**Verdict — MIXED**

The subsequent decline rate rose from 43.6% in the lowest-volume bucket to 57.0% for content with 100–499 March impressions. It remained elevated at 56.2% in the 500–2,999 bucket, then fell to 50.8% among content with at least 3,000 impressions.

March volume is therefore not a consistently increasing decline-risk signal. It can still support the baseline as an impact signal: reviewing a visible page may protect more search exposure even when volume alone does not imply greater decline risk. Every bucket has a large sample size.

### Signal 2 — CTR relative to search position

CTR must be interpreted alongside search position. I restrict this test to content with at least 100 March impressions so that a zero CTR reflects meaningful exposure rather than a tiny denominator.

Within each fixed position bucket, I compare content that received no clicks (`zero_ctr`) with content that received at least one click (`positive_ctr`). This tests a transparent version of the CTR-fix assumption: whether visible pages receiving no clicks show a different subsequent decline rate from pages at similar positions that receive clicks.

In [15]:
ctr_signal_frame = analysis_df[
    analysis_df["march_avg_position"].notna()
    & (analysis_df["march_avg_position"] > 0)
    & (analysis_df["march_impressions"] >= 100)
].copy()

position_labels = [
    "top_3",
    "page_1",
    "positions_11_20",
    "positions_21_50",
    "beyond_50",
]

ctr_signal_frame["position_bucket"] = pd.cut(
    ctr_signal_frame["march_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=position_labels,
    right=True,
)

ctr_signal_frame["ctr_group"] = np.where(
    ctr_signal_frame["march_ctr"] == 0,
    "zero_ctr",
    "positive_ctr",
)

ctr_position_table = (
    ctr_signal_frame
    .groupby(["position_bucket", "ctr_group"], observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_march_impressions=("march_impressions", "median"),
        median_march_ctr=("march_ctr", "median"),
        april_decline_rate=("is_declining_label", "mean"),
    )
    .reset_index()
)

ctr_position_table = ctr_position_table[
    ctr_position_table["n"] > 0
].copy()

ctr_position_table["median_march_ctr"] = (
    ctr_position_table["median_march_ctr"].round(3)
)

ctr_position_table["april_decline_rate"] = (
    ctr_position_table["april_decline_rate"] * 100
).round(1)

print("Signal 2 — Zero versus positive CTR within position buckets")
display(ctr_position_table)

Signal 2 — Zero versus positive CTR within position buckets


,position_bucket,ctr_group,n,median_march_impressions,median_march_ctr,april_decline_rate
0,top_3,positive_ctr,7856,2371.0,0.305,56.1
1,top_3,zero_ctr,2313,389.0,0.000,70.3
2,page_1,positive_ctr,34372,1705.0,0.306,49.6
3,page_1,zero_ctr,13076,310.0,0.000,67.3
4,positions_11_20,positive_ctr,11177,1078.0,0.304,51.8
5,positions_11_20,zero_ctr,8327,291.0,0.000,61.6
6,positions_21_50,positive_ctr,9649,2131.0,0.182,52.9
7,positions_21_50,zero_ctr,10067,276.0,0.000,55.9
8,beyond_50,positive_ctr,456,388.5,0.309,60.3
9,beyond_50,zero_ctr,3600,193.0,0.000,57.0


**Verdict — MIXED**

Among content ranking in positions 1–20, zero-CTR pages consistently had higher subsequent decline rates than positive-CTR pages in the same broad position bucket. The differences were 14.2 percentage points in the top three, 17.7 points across the rest of page one, and 9.8 points in positions 11–20.

The relationship weakened to 3.0 percentage points for positions 21–50 and reversed beyond position 50. The zero-CTR groups also had lower median impression volume, so this comparison does not isolate CTR as a causal factor.

The signal supports a restricted review rule for visible, zero-CTR content ranking within the top 20, but it does not support applying that rule across every search position.

### Final baseline rule

A content item enters the review queue when it meets all three conditions:

1. It received at least 100 impressions in March.
2. Its March average position was greater than 0 and no worse than 20.
3. It received zero March clicks, giving it a March CTR of zero.

Eligible items are ranked by March impressions. This prioritizes pages with more affected search exposure while keeping the rule transparent.

- **Score:** March impressions for eligible items
- **Reason code:** `visible_zero_ctr_top20`
- **Action label:** `review_title_and_meta`

The action is a recommendation for human review, not a claim that title or metadata changes will cause traffic recovery.



---






## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline applies the locked rule without fitted weights. Identifiers are retained only so the recommendations can be traced; they are not score inputs. The exported queue contains decision-time March fields only and excludes April outcomes and label-derived columns.

In [16]:
from pathlib import Path

scored_df = analysis_df.copy()

scored_df["is_eligible"] = (
    (scored_df["march_impressions"] >= 100)
    & (scored_df["march_avg_position"] > 0)
    & (scored_df["march_avg_position"] <= 20)
    & (scored_df["march_clicks"] == 0)
)

scored_df["baseline_score"] = np.where(
    scored_df["is_eligible"],
    scored_df["march_impressions"],
    0,
)

# The ranked action queue contains only items selected by the rule.
baseline_queue = (
    scored_df.loc[scored_df["is_eligible"]]
    .sort_values(
        ["baseline_score", "content_hash_id"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

baseline_queue["baseline_rank"] = np.arange(
    1, len(baseline_queue) + 1
)

baseline_queue["reason_code"] = "visible_zero_ctr_top20"
baseline_queue["action_label"] = "review_title_and_meta"

queue_columns = [
    "baseline_rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
]

baseline_queue = baseline_queue[queue_columns]

# Confirm that no future outcome or label is exported.
forbidden_export_columns = {
    "april_impressions",
    "is_declining_label",
}

assert forbidden_export_columns.isdisjoint(baseline_queue.columns)
assert baseline_queue["march_impressions"].ge(100).all()
assert baseline_queue["march_clicks"].eq(0).all()
assert baseline_queue["march_avg_position"].gt(0).all()
assert baseline_queue["march_avg_position"].le(20).all()
assert baseline_queue["baseline_rank"].is_unique
assert baseline_queue["reason_code"].nunique() == 1
assert baseline_queue["action_label"].nunique() == 1

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
baseline_queue.to_csv(output_path, index=False)

print(f"Eligible queue items: {len(baseline_queue):,}")
print(f"Wrote ranked queue to: {output_path}")
print(f"Highest baseline score: {baseline_queue['baseline_score'].max():,.0f}")
print(f"Lowest baseline score: {baseline_queue['baseline_score'].min():,.0f}")

# Safe preview: decision-time metrics only, without identifiers.
display(
    baseline_queue[
        [
            "baseline_rank",
            "baseline_score",
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

Eligible queue items: 23,716
Wrote ranked queue to: work/outputs/baseline_action_score.csv
Highest baseline score: 44,707
Lowest baseline score: 100


,baseline_rank,baseline_score,march_impressions,march_clicks,march_ctr,march_avg_position,reason_code,action_label
0,1,44707.0,44707.0,0.0,0.0,1.400049,visible_zero_ctr_top20,review_title_and_meta
1,2,38865.0,38865.0,0.0,0.0,4.804554,visible_zero_ctr_top20,review_title_and_meta
2,3,28950.0,28950.0,0.0,0.0,9.563005,visible_zero_ctr_top20,review_title_and_meta
3,4,24908.0,24908.0,0.0,0.0,3.431106,visible_zero_ctr_top20,review_title_and_meta
4,5,21519.0,21519.0,0.0,0.0,5.347182,visible_zero_ctr_top20,review_title_and_meta
5,6,19938.0,19938.0,0.0,0.0,10.909118,visible_zero_ctr_top20,review_title_and_meta
6,7,19292.0,19292.0,0.0,0.0,8.243572,visible_zero_ctr_top20,review_title_and_meta
7,8,17115.0,17115.0,0.0,0.0,6.012854,visible_zero_ctr_top20,review_title_and_meta
8,9,15902.0,15902.0,0.0,0.0,8.564017,visible_zero_ctr_top20,review_title_and_meta
9,10,14813.0,14813.0,0.0,0.0,3.130493,visible_zero_ctr_top20,review_title_and_meta


The top of the queue is reviewed skeptically rather than accepted automatically. For each of the first 20 recommendations, I compare the March evidence with the later April outcome and consider circumstances that could make the recommended action inappropriate.

April outcomes are attached only for retrospective evaluation. They were not used to calculate the baseline score or rank.

In [17]:
review_frame = baseline_queue.merge(
    analysis_df[
        [
            "client_hash_id",
            "content_hash_id",
            "april_impressions",
            "is_declining_label",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one",
)

review_frame["impression_change_pct"] = (
    100
    * (
        review_frame["april_impressions"]
        - review_frame["march_impressions"]
    )
    / review_frame["march_impressions"]
)

base_rate = analysis_df["is_declining_label"].mean()
precision_at_20 = review_frame.head(20)["is_declining_label"].mean()
precision_at_50 = review_frame.head(50)["is_declining_label"].mean()

print(f"Overall decline base rate: {base_rate:.1%}")
print(f"Baseline Precision@20: {precision_at_20:.1%}")
print(f"Baseline Precision@50: {precision_at_50:.1%}")

top20_review = review_frame.head(20)[
    [
        "baseline_rank",
        "march_impressions",
        "march_avg_position",
        "april_impressions",
        "impression_change_pct",
        "is_declining_label",
        "action_label",
    ]
].copy()

top20_review["march_avg_position"] = (
    top20_review["march_avg_position"].round(2)
)
top20_review["impression_change_pct"] = (
    top20_review["impression_change_pct"].round(1)
)

display(top20_review)

Overall decline base rate: 51.1%
Baseline Precision@20: 90.0%
Baseline Precision@50: 84.0%


,baseline_rank,march_impressions,march_avg_position,april_impressions,impression_change_pct,is_declining_label,action_label
0,1,44707.0,1.40,1289.0,-97.1,1,review_title_and_meta
1,2,38865.0,4.80,1180.0,-97.0,1,review_title_and_meta
2,3,28950.0,9.56,91524.0,216.1,0,review_title_and_meta
3,4,24908.0,3.43,13631.0,-45.3,1,review_title_and_meta
4,5,21519.0,5.35,517.0,-97.6,1,review_title_and_meta
5,6,19938.0,10.91,1355.0,-93.2,1,review_title_and_meta
6,7,19292.0,8.24,2953.0,-84.7,1,review_title_and_meta
7,8,17115.0,6.01,4283.0,-75.0,1,review_title_and_meta
8,9,15902.0,8.56,2169.0,-86.4,1,review_title_and_meta
9,10,14813.0,3.13,7083.0,-52.2,1,review_title_and_meta


### Baseline evaluation

The baseline achieved Precision@20 of 90.0% and Precision@50 of 84.0%, compared with an overall decline base rate of 51.1%. This means 18 of the top 20 and 42 of the top 50 recommendations met the defined April-decline outcome.

This is a retrospective baseline result, not evidence that the rule will generalize to another period. The rule was developed after examining signals in this same March–April slice. It is now frozen so that the Week 5 model can be compared against the same benchmark without moving the goalposts.



---



## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Skeptical top-20 review

| Rank | Confidence | Action and why it is here | What would make it wrong |
|---:|---|---|---|
| 1 | High | **Review title and metadata.** It had 44,707 impressions, position 1.40, zero clicks, and later declined 97.1%. | A tracking or aggregation failure could explain the implausible combination of top position, heavy exposure, and zero clicks. |
| 2 | High | **Review title and metadata.** It had 38,865 impressions at position 4.80 with zero clicks, followed by a 97.0% decline. | The decline could reflect deindexing, seasonality, or reporting loss rather than a weak search snippet. |
| 3 | Low | **Review title and metadata.** It entered because 28,950 impressions at position 9.56 produced zero clicks. | This was a false positive: April impressions increased 216.1%, so the March pattern did not indicate continuing decline. |
| 4 | High | **Review title and metadata.** It received 24,908 impressions at position 3.43 without clicks and later declined 45.3%. | A title change would be inappropriate if the decline came from changing demand, indexing, or measurement problems. |
| 5 | High | **Review title and metadata.** It had 21,519 impressions at position 5.35 with zero clicks, followed by a 97.6% decline. | Near-total loss may indicate removal, deindexing, or incomplete April coverage rather than a CTR problem. |
| 6 | High | **Review title and metadata.** It had 19,938 impressions at position 10.91, zero clicks, and a later 93.2% decline. | The recommendation would be wrong if the page intentionally serves low-click informational queries or lost visibility for unrelated reasons. |
| 7 | High | **Review title and metadata.** It had 19,292 impressions at position 8.24 without clicks and later declined 84.7%. | A reporting mismatch or a major change in the page's query mix could make the CTR-based diagnosis wrong. |
| 8 | High | **Review title and metadata.** It received 17,115 impressions at position 6.01 with zero clicks and later declined 75.0%. | The decline may be seasonal or caused by SERP changes that cannot be corrected through title and metadata edits. |
| 9 | High | **Review title and metadata.** It had 15,902 impressions at position 8.56 without clicks, followed by an 86.4% decline. | The action would be wrong if impressions and clicks were measured inconsistently or the page was intentionally retired. |
| 10 | High | **Review title and metadata.** It had 14,813 impressions at position 3.13, zero clicks, and a later 52.2% decline. | Strong position with zero clicks may be a data-quality issue rather than evidence of an unattractive snippet. |
| 11 | High | **Review title and metadata.** It received 14,482 impressions at position 7.40 without clicks and later declined 56.5%. | A shift in search demand or query intent could explain the decline without the title or metadata being defective. |
| 12 | High | **Review title and metadata.** It had 14,372 impressions at position 7.98, zero clicks, and a subsequent 72.0% decline. | The recommendation would be wrong if tracking failed or the loss came from indexing rather than click appeal. |
| 13 | High | **Review title and metadata.** It had 13,001 impressions at position 6.62 without clicks and later declined 99.0%. | Such an extreme loss could mean deletion, deindexing, or incomplete April reporting, requiring diagnosis before editing. |
| 14 | High | **Review title and metadata.** It received 12,588 impressions at position 1.14 with zero clicks and later declined 98.8%. | The top-position/zero-click combination is suspicious enough that a measurement or aggregation error could invalidate the recommendation. |
| 15 | High | **Review title and metadata.** It had 11,355 impressions at position 6.87 without clicks and later declined 81.6%. | The decline could be driven by lower demand or SERP features rather than something editable on the page. |
| 16 | Low | **Review title and metadata.** It entered because 11,344 impressions at position 2.01 generated zero clicks. | This was a false positive: April impressions increased 6.4%, so the expected decline did not occur. |
| 17 | High | **Review title and metadata.** It had 11,187 impressions at position 4.62, zero clicks, and a later 63.5% decline. | The action would be wrong if the page lost impressions because of seasonality, deindexing, or a changed query mix. |
| 18 | High | **Review title and metadata.** It received 10,886 impressions at position 2.96 without clicks and later declined 49.0%. | A reporting anomaly could explain zero clicks at such a strong average position, so the raw GSC pattern should be checked first. |
| 19 | Medium | **Review title and metadata.** It had 10,462 impressions, a reported position of 0.23, zero clicks, and a later 51.5% decline. | The unusually low position value suggests a possible interpretation or data-quality issue that should be resolved before action. |
| 20 | High | **Review title and metadata.** It had 10,245 impressions at position 10.43 without clicks and later declined 66.9%. | The recommendation would be wrong if the decline reflects demand loss or indexing rather than poor title and metadata performance. |



---



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The top-20 review found recommendations that did not meet the later decline outcome. These false positives are useful because they show where the simple rule can fail.

I also reconstruct the score from its declared March-only inputs and check the exported columns. Identifiers may be used for traceability and deterministic tie-breaking, but they are not score features.

In [18]:
weak_picks = (
    review_frame.loc[review_frame["is_declining_label"] == 0]
    .head(10)
    .copy()
)

weak_picks["impression_change_pct"] = (
    100
    * (
        weak_picks["april_impressions"]
        - weak_picks["march_impressions"]
    )
    / weak_picks["march_impressions"]
)

weak_pick_columns = [
    "baseline_rank",
    "march_impressions",
    "march_avg_position",
    "march_ctr",
    "april_impressions",
    "impression_change_pct",
    "action_label",
]

weak_pick_display = weak_picks[weak_pick_columns].copy()
weak_pick_display["march_avg_position"] = (
    weak_pick_display["march_avg_position"].round(2)
)
weak_pick_display["impression_change_pct"] = (
    weak_pick_display["impression_change_pct"].round(1)
)

print("Highest-ranked false positives")
display(weak_pick_display)

# Reconstruct the rule using only its declared March inputs.
declared_score_inputs = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
]

reconstructed_eligibility = (
    (scored_df["march_impressions"] >= 100)
    & (scored_df["march_clicks"] == 0)
    & (scored_df["march_avg_position"] > 0)
    & (scored_df["march_avg_position"] <= 20)
)

reconstructed_score = np.where(
    reconstructed_eligibility,
    scored_df["march_impressions"],
    0,
)

forbidden_input_names = {
    "april_impressions",
    "is_declining_label",
    "trend_pct",
    "trend_direction",
}

score_reproduces_exactly = np.array_equal(
    reconstructed_score,
    scored_df["baseline_score"].to_numpy(),
)

no_forbidden_score_inputs = forbidden_input_names.isdisjoint(
    declared_score_inputs
)

no_future_fields_exported = forbidden_input_names.isdisjoint(
    baseline_queue.columns
)

print(f"Declared score inputs: {declared_score_inputs}")
print(f"Score reconstructs exactly: {score_reproduces_exactly}")
print(f"No future or label-derived score inputs: {no_forbidden_score_inputs}")
print(f"No future or label-derived fields in exported queue: {no_future_fields_exported}")

assert score_reproduces_exactly
assert no_forbidden_score_inputs
assert no_future_fields_exported

Highest-ranked false positives


,baseline_rank,march_impressions,march_avg_position,march_ctr,april_impressions,impression_change_pct,action_label
2,3,28950.0,9.56,0.0,91524.0,216.1,review_title_and_meta
15,16,11344.0,2.01,0.0,12075.0,6.4,review_title_and_meta
24,25,9312.0,5.83,0.0,14439.0,55.1,review_title_and_meta
28,29,8258.0,5.47,0.0,7080.0,-14.3,review_title_and_meta
29,30,8232.0,7.91,0.0,31221.0,279.3,review_title_and_meta
30,31,8220.0,18.57,0.0,9087.0,10.5,review_title_and_meta
39,40,7329.0,1.39,0.0,8060.0,10.0,review_title_and_meta
43,44,6827.0,1.00,0.0,9513.0,39.3,review_title_and_meta
51,52,6358.0,14.37,0.0,6380.0,0.3,review_title_and_meta
53,54,6284.0,7.17,0.0,6467.0,2.9,review_title_and_meta


Declared score inputs: ['march_impressions', 'march_clicks', 'march_avg_position']
Score reconstructs exactly: True
No future or label-derived score inputs: True
No future or label-derived fields in exported queue: True


### Weak-pick interpretation

The highest-ranked false positive was rank 3. Despite receiving zero March clicks from 28,950 impressions at position 9.56, its impressions increased by 216.1% in April. Other false positives increased by as much as 279.3%. This shows that a zero-CTR snapshot can be temporary, affected by changing query demand, or reflect a measurement issue rather than a continuing decline.

Rank 29 decreased by 14.3% but is counted as a false positive because it did not cross the predefined decline threshold of more than 15%. This illustrates that binary evaluation can treat a near-threshold recommendation as entirely incorrect.

The baseline is therefore useful for surfacing unusual high-exposure pages, but the action still requires human diagnosis. It cannot determine that title or metadata caused the observed pattern.

The leakage checks passed. The score was reproduced exactly from March impressions, clicks, and average position. No April outcome, label-derived field, future-window field, product flag, or identifier was used as a score input. The exported queue also excludes future outcomes and labels.



---



###Metrics JSONs

In [19]:
import json

metrics = {
    "assignment": "w04_baseline_score",
    "observation_window": "2026-03",
    "outcome_window": "2026-04",
    "sealed_test_month": "2026-06",
    "evaluation_rows": int(len(analysis_df)),
    "eligible_queue_items": int(len(baseline_queue)),
    "decline_threshold": -0.15,
    "overall_decline_base_rate": round(float(base_rate), 4),
    "precision_at_20": round(float(precision_at_20), 4),
    "precision_at_50": round(float(precision_at_50), 4),
    "signal_verdicts": {
        "march_impression_volume": "MIXED",
        "position_adjusted_ctr": "MIXED",
    },
    "rule": {
        "minimum_march_impressions": 100,
        "minimum_march_avg_position_exclusive": 0,
        "maximum_march_avg_position_inclusive": 20,
        "required_march_clicks": 0,
        "score": "march_impressions for eligible items; zero otherwise",
        "reason_code": "visible_zero_ctr_top20",
        "action_label": "review_title_and_meta",
    },
    "leakage_checks": {
        "score_reconstructs_exactly": bool(score_reproduces_exactly),
        "no_future_or_label_derived_score_inputs": bool(
            no_forbidden_score_inputs
        ),
        "no_future_or_label_derived_export_fields": bool(
            no_future_fields_exported
        ),
    },
    "evaluation_note": (
        "Retrospective March-April result on the development slice; "
        "not out-of-sample performance."
    ),
}

metrics_path = Path("work/outputs/baseline_metrics.json")

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(f"Wrote metrics receipt to: {metrics_path}")
print(json.dumps(metrics, indent=2))

Wrote metrics receipt to: work/outputs/baseline_metrics.json
{
  "assignment": "w04_baseline_score",
  "observation_window": "2026-03",
  "outcome_window": "2026-04",
  "sealed_test_month": "2026-06",
  "evaluation_rows": 158549,
  "eligible_queue_items": 23716,
  "decline_threshold": -0.15,
  "overall_decline_base_rate": 0.5108,
  "precision_at_20": 0.9,
  "precision_at_50": 0.84,
  "signal_verdicts": {
    "march_impression_volume": "MIXED",
    "position_adjusted_ctr": "MIXED"
  },
  "rule": {
    "minimum_march_impressions": 100,
    "minimum_march_avg_position_exclusive": 0,
    "maximum_march_avg_position_inclusive": 20,
    "required_march_clicks": 0,
    "score": "march_impressions for eligible items; zero otherwise",
    "reason_code": "visible_zero_ctr_top20",
    "action_label": "review_title_and_meta"
  },
  "leakage_checks": {
    "score_reconstructs_exactly": true,
    "no_future_or_label_derived_score_inputs": true,
    "no_future_or_label_derived_export_fields": true
  

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.